In [93]:
import pandas as pd
import numpy as np
import torch as t

In [94]:
def split_at_nans(data:pd.DataFrame) -> list[pd.DataFrame] :
    '''splits data up into chunks whenever there is a block of nan values'''

    # bool mask of if there is a nan value on each row
    na_mask : pd.DataFrame = data.isna()
    na_mask = np.sum(na_mask,axis=1)
    na_mask = na_mask.astype(bool)

    def block_search(data:pd.DataFrame,indexes:slice) -> list[slice] :
        '''finds blocks of True in dataframe using binary search'''
        start, stop = indexes.start, indexes.stop

        # base case
        if start+1 == stop :
            return [indexes] if data.iloc[start] else []

        mid = start + (stop-start)//2
        left, right = slice(start,mid,1), slice(mid,stop,1)

        # left side
        left_datum = na_mask.iloc[left.start]
        if left_datum :
            homogenous = np.all(na_mask.iloc[left])
        else :
            homogenous = np.all(~na_mask.iloc[left])

        if homogenous :
            left_blocks = [left] if left_datum else []
        else :
            left_blocks = block_search(data,left)

        # right side
        right_datum = na_mask.iloc[right.start]
        if right_datum :
            homogenous = np.all(na_mask.iloc[right])
        else :
            homogenous = np.all(~na_mask.iloc[right])
        
        if homogenous :
            right_blocks = [right] if right_datum else []
        else :
            right_blocks = block_search(data,right)

        # combine blocks
        if left_blocks == [] :
            return right_blocks
        elif right_blocks == [] :
            return left_blocks
        
        if left_blocks[-1].stop == right_blocks[0].start :
            # combine slices
            left_slice = left_blocks.pop(-1)
            right_slice = right_blocks.pop(0)
            left_blocks.append(slice(left_slice.start,right_slice.stop))

        return left_blocks + right_blocks
    
    nan_blocks = block_search(data,slice(0,data.shape[0]))

    data_blocks : list[slice] = []

    # convert nan_blocks to data blocks by genorating inverse slices
    temp = sum([[s.start,s.stop] for s in nan_blocks],start=[])
    temp.insert(0,0)
    temp.append(data.shape[0])
    
    for i in range(0,len(temp),2) :
        s = slice(temp[i],temp[i+1],1)
        data_blocks.append(data.iloc[s,:])

    return data_blocks

In [75]:
example_data = pd.read_csv('../data/2/exp02H20141127_16h29.csv')
example_data.head(8)

,X1,Y1,H1,X2,Y2,H2
0,-210.5314,-206.7752,-35.28264,-101.6321,-262.6715,-28.46980
1,-211.6309,-205.6295,-35.28648,-104.7569,-262.1026,-28.47645
2,-213.4879,-204.2483,-35.28863,-108.5043,-261.2918,-28.48702
3,-215.0812,-202.6719,-35.28820,-110.7450,-260.8367,-28.47787
4,-216.5786,-201.3576,-35.29496,-114.9723,-259.9251,-28.48686
5,-218.0429,-199.9651,-35.29347,-116.9992,-259.4808,-28.49900
6,-220.0927,-198.3625,-35.29278,-119.7869,-258.8347,-28.49305
7,-221.2513,-197.1864,-35.29315,-121.5687,-258.3917,-28.49902


In [76]:
type(example_data.shape[0])

int

In [143]:
diffs = example_data.diff()

v1 = np.sqrt(diffs['X1']**2 + diffs['Y1']**2)
h1 = diffs['H1']

v2 = np.sqrt(diffs['X2']**2 + diffs['Y2']**2)
h2 = diffs['H2']

example_data['V1'] = v1
example_data['V2'] = v2
example_data['AV1'] = h1
example_data['AV2'] = h2

# print(example_data.head(5))
# print(example_data.shape)

target_df = example_data[['V1', 'V2', 'AV1', 'AV2']]
target_df = split_at_nans(target_df)[0]
target_df = target_df.to_numpy()
print(target_df.shape)

(40770, 4)


In [144]:
def create_sequences(data, window_size = 30):

    shape = data.shape

    X = np.zeros((shape[1], shape[0]-window_size, window_size))
    y = np.zeros((shape[1], shape[0]-window_size))

    for c in range(shape[1]):

        for i in range(len(data) - window_size):

            X[c,i] = data[i : i + window_size, c]
            
            y[c,i] = data[i + window_size, c]

    X = X.transpose(1, 2, 0)
    
    y = y.transpose(1, 0)

    return X, y

In [145]:
print(type(target_df))
X, y = create_sequences(data = target_df)

<class 'numpy.ndarray'>


In [146]:
print(X.shape)
print(y.shape)

(40740, 30, 4)
(40740, 4)


In [151]:
from sklearn.preprocessing import StandardScaler

def prepare_fish_data(target_df, window_size=30, train_pct=0.7, val_pct=0.15):
    '''
    Splits the data into training, validation and testing by percentage. 
    Applies a scalar to normalise the data with mean 0 and std 1
    returns the splits for X and y, (sequence and next value) and the scalar
    '''
    n = len(target_df)
    train_end = int(n * train_pct)
    val_end = int(n * (train_pct + val_pct))
    
    raw_train = target_df[:train_end]
    raw_val = target_df[train_end:val_end]
    raw_test = target_df[val_end:]
    
    scaler = StandardScaler() # Found online, pretty easy to use and makes training much more efficent.
    train_scaled = scaler.fit_transform(raw_train)
    val_scaled = scaler.transform(raw_val)
    test_scaled = scaler.transform(raw_test)
    
    X_train, y_train = create_sequences(train_scaled, window_size)
    X_val, y_val = create_sequences(val_scaled, window_size)
    X_test, y_test = create_sequences(test_scaled, window_size)
    
    return X_train, y_train, X_val, y_val, X_test, y_test, scaler

X_train, y_train, X_val, y_val, X_test, y_test, scaler = prepare_fish_data(target_df)

print(f"Train: {X_train.shape}, {y_train.shape}")
print(f"Val:   {X_val.shape}, {y_val.shape}")
print(f"Test:  {X_test.shape}, {y_test.shape}")

Train: (28509, 30, 4), (28509, 4)
Val:   (6085, 30, 4), (6085, 4)
Test:  (6086, 30, 4), (6086, 4)
